# Customer Support Agent — Capstone Lab 2 (Extended)

```
                    ┌─────────────────────┐
                    │  User Query         │
                    └──────────┬──────────┘
                               │
                               ▼
                    ┌─────────────────────┐
                    │ Router Agent        │
                    │ (LLM Intent Detect) │◄── Prompt Cache
                    └──────────┬──────────┘
                               │
               ┌───────────────┼────────────────┐
               │               │                │
               ▼               ▼                ▼
      ┌──────────────┐ ┌─────────────┐ ┌──────────────┐
      │ FAQ Tool     │ │ Ticket Tool │ │ Order Tool   │
      │ (Vector DB)  │ │ (Async MQ)  │ │ API Tool     │
      │  + Retries   │ │  + Retries  │ │  + Retries   │
      └──────┬───────┘ └──────┬──────┘ └──────┬───────┘
             │                │               │
             └────────┬───────┴───────┬───────┘
                      │               │
                      ▼               ▼
          ┌─────────────────────┐
          │ Chaining Layer      │
          │ Multi-step Reasoning│
          └──────────┬──────────┘
                     │
                     ▼
          ┌─────────────────────┐
          │ Approval Gate       │
          │ Human Review        │
          └──────────┬──────────┘
                     │
                     ▼
          ┌─────────────────────┐
          │ Tracer / Logger     │
          │ (OpenTelemetry)     │
          └──────────┬──────────┘
                     │
                     ▼
          ┌─────────────────────┐
          │ Final Response      │
          └─────────────────────┘
```

## 1. Setup and Infrastructure
Install all dependencies including `tenacity` for retries and `opentelemetry` for tracing.

In [1]:
!pip install -q \
    anthropic \
    tenacity \
    opentelemetry-api \
    opentelemetry-sdk \
    faiss-cpu \
    sentence-transformers

In [2]:
import os
import uuid
import time
import json
import random
import functools
from typing import Any, Dict, List, Optional
from datetime import datetime

# ── Retry ──────────────────────────────────────────────────────────────────
from tenacity import (
    retry,
    stop_after_attempt,
    wait_exponential,
    retry_if_exception_type,
    before_sleep_log,
)
import logging

# ── OpenTelemetry Tracing ───────────────────────────────────────────────────
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import (
    SimpleSpanProcessor,
    ConsoleSpanExporter,
)
from opentelemetry.sdk.resources import Resource

# ── Anthropic ───────────────────────────────────────────────────────────────
import anthropic

logging.basicConfig(level=logging.WARNING)
log = logging.getLogger("support_agent")

print("All imports OK.")

All imports OK.


## 2. Tracer Setup (OpenTelemetry)
Every span is printed to the console so you can see the full call tree.

In [3]:
# ── Configure a tracer that writes spans to stdout ─────────────────────────
resource = Resource.create({"service.name": "support-agent"})
provider = TracerProvider(resource=resource)
provider.add_span_processor(SimpleSpanProcessor(ConsoleSpanExporter()))
trace.set_tracer_provider(provider)
tracer = trace.get_tracer("support_agent")

# ── Convenience decorator: wrap any function in a span ─────────────────────
def traced(span_name: Optional[str] = None):
    """Decorator: automatically wraps a function in an OTel span."""
    def decorator(fn):
        @functools.wraps(fn)
        def wrapper(*args, **kwargs):
            name = span_name or fn.__name__
            with tracer.start_as_current_span(name) as span:
                span.set_attribute("function", fn.__name__)
                try:
                    result = fn(*args, **kwargs)
                    span.set_attribute("status", "ok")
                    return result
                except Exception as exc:
                    span.record_exception(exc)
                    span.set_attribute("status", "error")
                    raise
        return wrapper
    return decorator

print("Tracer configured (spans → stdout).")

Tracer configured (spans → stdout).


## 3. Mock Infrastructure (Redis + Vector Store)

In [4]:
class MockRedis:
    """In-process short-term memory store (replaces Redis in this env)."""
    def __init__(self): self._store: Dict[str, Any] = {}
    def set(self, key: str, val: Any): self._store[key] = val
    def get(self, key: str): return self._store.get(key)
    def delete(self, key: str): self._store.pop(key, None)


class MockVectorStore:
    """Append-only in-memory store — mimics a vector DB for long-term recall."""
    def __init__(self): self._docs: List[str] = []

    def add_text(self, text: str):
        self._docs.append(text)

    def search(self, query: str, k: int = 3) -> List[str]:
        # Return the k most recent docs as a simple recall proxy
        return self._docs[-k:] if self._docs else []


redis_mem   = MockRedis()
vector_db   = MockVectorStore()
print("Infrastructure initialised.")

Infrastructure initialised.


## 4. Prompt Cache
The system prompt is built once and reused across every LLM call.  
With the Anthropic SDK you signal caching by adding `cache_control: {"type": "ephemeral"}`  
to the last large system-prompt block — the API then serves subsequent calls from its cache,  
cutting latency and token cost.

In [5]:
# ── Build the (large) system prompt once — mark it for caching ─────────────
SYSTEM_PROMPT_BLOCKS = [
    {
        "type": "text",
        "text": (
            "You are a customer-support routing agent for TechCorp.\n"
            "Your ONLY job is to classify the user's query into exactly ONE of:\n"
            "  • order_tool   — questions about order status / tracking\n"
            "  • ticket_tool  — complaints, problems, or issues that need a human\n"
            "  • faq_tool     — general product/policy questions\n"
            "\n"
            "Reply with a single JSON object and nothing else:\n"
            '  {"intent": "<tool_name>", "reason": "<one sentence>"}\n'
            "\n"
            "--- Company knowledge base (for context only) ---\n"
            "Return policy: items can be returned within 30 days with receipt.\n"
            "Shipping: standard 5-7 business days; express 1-2 business days.\n"
            "Warranty: 1-year limited warranty on all electronics.\n"
            "Support hours: Mon-Fri 9am-6pm EST.\n"
            "--- End knowledge base ---"
        ),
        # ↓ This marks the block for Anthropic prompt caching
        "cache_control": {"type": "ephemeral"},
    }
]

print("System-prompt cache block ready.")

System-prompt cache block ready.


## 5. Tool Definitions with Retries
Every tool is wrapped with `@retry` from **tenacity**:  
exponential back-off, up to 3 attempts, only on transient `IOError` / `RuntimeError`.

In [6]:
# ── Helper: simulate occasional transient failures ─────────────────────────
def _maybe_fail(tool_name: str, fail_rate: float = 0.3):
    """Randomly raises RuntimeError to simulate a flaky upstream service."""
    if random.random() < fail_rate:
        raise RuntimeError(f"[{tool_name}] Transient upstream error — retrying…")


# ── Retry policy shared by all tools ──────────────────────────────────────
RETRY_POLICY = dict(
    stop=stop_after_attempt(3),
    wait=wait_exponential(multiplier=0.5, min=0.5, max=4),
    retry=retry_if_exception_type((RuntimeError, IOError)),
    reraise=True,
)


# ── Tool 1: FAQ (Vector DB search) ────────────────────────────────────────
@traced("tool.faq")
@retry(**RETRY_POLICY)
def faq_tool(query: str) -> str:
    """Search the knowledge base for policy / product answers."""
    _maybe_fail("faq_tool")
    past = vector_db.search(query)
    context = " | ".join(past) if past else "No prior context."
    return (
        f"FAQ Result for '{query}':\n"
        f"  Policy: Please refer to our return policy on the website.\n"
        f"  Prior context: {context}"
    )


# ── Tool 2: Order Status (API call) ───────────────────────────────────────
@traced("tool.order")
@retry(**RETRY_POLICY)
def order_tool(order_id: str) -> str:
    """Check the status of an order via an external API."""
    _maybe_fail("order_tool")
    statuses = ["PROCESSING", "SHIPPED", "OUT_FOR_DELIVERY", "DELIVERED"]
    status = random.choice(statuses)
    return f"Order {order_id} is currently: {status}."


# ── Tool 3: Ticket (Async Queue) ──────────────────────────────────────────
@traced("tool.ticket")
@retry(**RETRY_POLICY)
def ticket_tool_async(issue: str) -> Dict[str, str]:
    """Enqueue a support ticket on the async message queue."""
    _maybe_fail("ticket_tool", fail_rate=0.2)
    ticket_id = str(uuid.uuid4())[:8].upper()
    enqueued_at = datetime.utcnow().isoformat()
    print(f"  [Queue] ✓ Ticket {ticket_id} enqueued at {enqueued_at}: {issue}")
    return {"status": "queued", "ticket_id": ticket_id, "enqueued_at": enqueued_at}


print("Tools defined (with retries + tracing).")

Tools defined (with retries + tracing).


## 6. LLM Router (Anthropic Claude with Prompt Caching)
The router sends the user query to Claude and parses the JSON intent response.

In [7]:
# Set your key here or load from env
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."
client = anthropic.Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY", "YOUR_KEY_HERE"))


@traced("router.llm_classify")
@retry(**RETRY_POLICY)
def llm_router(query: str, conversation_history: List[Dict] = None) -> Dict[str, str]:
    """
    Call Claude to classify the user's intent.
    Uses prompt caching on the system block to reduce cost on repeated calls.
    Falls back to keyword routing if the API key is not set.
    """
    # ── Fallback: keyword routing (used when no API key is available) ──────
    api_key = os.environ.get("ANTHROPIC_API_KEY", "YOUR_KEY_HERE")
    if api_key == "YOUR_KEY_HERE" or not api_key.startswith("sk-"):
        print("  [Router] No API key — using keyword fallback.")
        q = query.lower()
        if "order" in q or "track" in q or "ship" in q:
            return {"intent": "order_tool", "reason": "Query mentions order/tracking."}
        elif "problem" in q or "issue" in q or "broken" in q or "complaint" in q:
            return {"intent": "ticket_tool", "reason": "Query describes a problem."}
        else:
            return {"intent": "faq_tool", "reason": "General question routed to FAQ."}

    # ── LLM routing ────────────────────────────────────────────────────────
    messages = (conversation_history or []) + [
        {"role": "user", "content": query}
    ]

    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=256,
        system=SYSTEM_PROMPT_BLOCKS,   # ← cached system prompt
        messages=messages,
    )

    # Log cache hit/miss from usage metadata
    usage = response.usage
    cache_read   = getattr(usage, "cache_read_input_tokens",   0)
    cache_create = getattr(usage, "cache_creation_input_tokens", 0)
    print(f"  [Cache] read={cache_read} | created={cache_create} | input={usage.input_tokens}")

    raw = response.content[0].text.strip()
    # Strip markdown code fences if present
    raw = raw.replace("```json", "").replace("```", "").strip()
    return json.loads(raw)


print("LLM router ready.")

LLM router ready.


## 7. Support Agent — Routing + Chaining + Human Approval Gate

In [8]:
class SupportAgent:
    """
    Full support agent with:
      • Short-term memory  (Redis)
      • Long-term recall   (Vector DB)
      • LLM routing        (Claude, prompt-cached)
      • Tool execution     (FAQ / Order / Ticket, all with retries)
      • Human approval gate (for destructive / ticket actions)
      • OTel tracing       (every step is a span)
    """

    TOOLS_NEEDING_APPROVAL = {"ticket_tool"}

    def __init__(self, user_id: str, auto_approve: bool = True):
        self.user_id      = user_id
        self.auto_approve = auto_approve   # set False for interactive approval
        self._conversation: List[Dict] = []

    # ── Human-approval gate ────────────────────────────────────────────────
    @traced("gate.human_approval")
    def _approval_gate(self, intent: str, query: str) -> bool:
        if intent not in self.TOOLS_NEEDING_APPROVAL:
            return True
        print(f"\n  [Approval Gate] ⚠  Action '{intent}' requires human review.")
        print(f"  [Approval Gate]    Query: \"{query}\"")
        if self.auto_approve:
            print("  [Approval Gate] ✓ Auto-approved (auto_approve=True).")
            return True
        answer = input("  Approve? (y/n): ").strip().lower()
        approved = answer == "y"
        print(f"  [Approval Gate] {'✓ Approved' if approved else '✗ Rejected'} by human.")
        return approved

    # ── Main request handler ───────────────────────────────────────────────
    @traced("agent.process_request")
    def process_request(self, query: str) -> Any:
        span = trace.get_current_span()
        span.set_attribute("user_id", self.user_id)
        span.set_attribute("query",   query)

        print(f"\n{'='*60}")
        print(f"User [{self.user_id}]: {query}")
        print(f"{'='*60}")

        # 1. Short-term memory: what did the user last do?
        last_intent = redis_mem.get(f"{self.user_id}:last_intent")
        if last_intent:
            print(f"  [Memory] Last intent: {last_intent}")

        # 2. Long-term recall: inject recent conversation snippets
        past_docs = vector_db.search(query)
        if past_docs:
            print(f"  [VectorDB] Recalled {len(past_docs)} past snippet(s).")

        # 3. LLM routing (with prompt caching)
        print("  [Router] Classifying intent…")
        routing = llm_router(query, conversation_history=self._conversation)
        intent  = routing["intent"]
        reason  = routing["reason"]
        print(f"  [Router] Intent='{intent}' | Reason: {reason}")
        span.set_attribute("intent", intent)

        # 4. Human approval gate
        approved = self._approval_gate(intent, query)
        if not approved:
            return {"status": "rejected", "reason": "Human reviewer rejected this action."}

        # 5. Tool execution (retries handled inside each tool)
        print(f"  [Executor] Calling {intent}…")
        try:
            if intent == "order_tool":
                # Chaining: extract order ID from query if present, else default
                import re
                match = re.search(r'[A-Z]{2,}-?\d{3,}', query.upper())
                order_id = match.group(0) if match else "ORD-123"
                result = order_tool(order_id)
            elif intent == "ticket_tool":
                result = ticket_tool_async(query)
            else:
                result = faq_tool(query)
        except RuntimeError as exc:
            # All retries exhausted
            span.record_exception(exc)
            result = {"status": "error", "detail": str(exc)}

        # 6. Memory updates
        redis_mem.set(f"{self.user_id}:last_intent", intent)
        vector_db.add_text(
            f"[{datetime.utcnow().isoformat()}] user='{self.user_id}' "
            f"query='{query}' intent='{intent}'"
        )
        # Update conversation history for multi-turn context
        self._conversation.append({"role": "user",      "content": query})
        self._conversation.append({"role": "assistant", "content": json.dumps(routing)})

        print(f"  [Result] {result}")
        return result


print("SupportAgent class ready.")

SupportAgent class ready.


## 8. Run the Agent
Three test queries covering all three tools.

In [9]:
agent = SupportAgent(user_id="user_456", auto_approve=True)

queries = [
    "Where is my order ORD-789?",
    "What is your return policy?",
    "I have a problem — my laptop screen is broken!",
]

for q in queries:
    agent.process_request(q)
    print()


User [user_456]: Where is my order ORD-789?
  [Router] Classifying intent…
  [Router] No API key — using keyword fallback.
{
    "name": "router.llm_classify",
    "context": {
        "trace_id": "0xcfcc7b296d6d4f8370fd2f3313b5db14",
        "span_id": "0x699108a37b52bb7e",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0xd13a79402226d6f3",
    "start_time": "2026-06-24T02:37:08.508013Z",
    "end_time": "2026-06-24T02:37:08.508013Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "function": "llm_router",
        "status": "ok"
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.34.1",
            "service.name": "support-agent"
        },
        "schema_url": ""
    }
}
  [Router] Intent='order_tool' | Reason: Query mentions order/tracking.

## 9. Verify Memory Persistence

In [10]:
print("=== Short-term memory (Redis) ===")
print("Last intent:", redis_mem.get("user_456:last_intent"))

print("\n=== Long-term memory (Vector DB) ===")
for i, doc in enumerate(vector_db.search("any", k=10), 1):
    print(f"  {i}. {doc}")

=== Short-term memory (Redis) ===
Last intent: ticket_tool

=== Long-term memory (Vector DB) ===
  1. [2026-06-24T02:37:08.511011] user='user_456' query='Where is my order ORD-789?' intent='order_tool'
  2. [2026-06-24T02:37:08.516850] user='user_456' query='What is your return policy?' intent='faq_tool'
  3. [2026-06-24T02:37:08.521023] user='user_456' query='I have a problem — my laptop screen is broken!' intent='ticket_tool'


## 10. Multi-turn Chaining Demo
The agent remembers what was said earlier in the session.

In [11]:
agent2 = SupportAgent(user_id="user_789", auto_approve=True)

# Turn 1: ask about an order
agent2.process_request("Can you check order ORD-999 for me?")

# Turn 2: follow-up (agent should have context from turn 1)
agent2.process_request("And what if I want to return it?")

print("\nConversation history stored:", len(agent2._conversation), "messages")


User [user_789]: Can you check order ORD-999 for me?
  [VectorDB] Recalled 3 past snippet(s).
  [Router] Classifying intent…
  [Router] No API key — using keyword fallback.
{
    "name": "router.llm_classify",
    "context": {
        "trace_id": "0x591d87fe82463bb9902d726a56849b9d",
        "span_id": "0x8836db37e5953640",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x063ddfe8d665ae5e",
    "start_time": "2026-06-24T02:37:08.572067Z",
    "end_time": "2026-06-24T02:37:08.572067Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "function": "llm_router",
        "status": "ok"
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.34.1",
            "service.name": "support-agent"
        },
        "schema_url": ""
    }
}
  [Router] Intent='or